# 09 — KRAS G12C 共有結合ドッキングチュートリアル
# KRAS G12C Covalent Docking Tutorial

**ターゲット**: KRAS G12C（Cys12 Switch II ポケット）  
**事例**: ARS-853 (ツール化合物, 6N2J) → Sotorasib / AMG-510 (FDA 承認, 6OIM)  
**ウォーヘッド**: Acrylamide（Michael acceptor → Cys12 SG）

---

## なぜ KRAS は長年「創薬不能 (undruggable)」とされたか

```
野生型 KRAS:  Gly12 ─── GDP 結合ポケット (球状・平坦)
              ↑ 小さすぎて有機分子が結合しにくい

G12C 変異  :  Cys12 SG が突出 → Switch II ポケット (SW2P) が開口
              ↑ 共有結合形成で不可逆的にロック可能！
```

→ G12C 変異が偶発的に**新たな共有結合標的**を生み出した。

---

## ARS-853 → Sotorasib の進化

| 項目 | ARS-853 (6N2J) | Sotorasib (6OIM) |
|------|----------------|------------------|
| 開発段階 | ツール化合物 (非臨床) | **FDA 承認 (2021, NSCLC)** |
| 分解能 | 2.4 Å | **1.9 Å** |
| SW2P 占有 | 部分的 | 完全占有 |
| His95 相互作用 | △ | ✓ (π-π) |
| 意義 | 概念実証 (G12C が標的になる証明) | 最終臨床形 |

---

## このノートブックで学べること

1. KRAS G12C の Switch II ポケット（SW2P）と GDP 共存構造の前処理
2. Cys12 SG の PDB からの検出と UniDock2 covalent docking 設定
3. `GridBox` の手動補正 — shallow pocket への対応
4. `validate_poses_posebusters()` によるポーズ妥当性確認
5. `plot_pareto_2d()` によるスコア vs 歪みエネルギーの多目的評価

> **Note**: ドッキング実行（Section 6）は UniDock2 + GPU が必要です。

In [ ]:
# CONFIG -----------------------------------------------------------------------
DATA_DIR     = "../data/kras_g12c"    # PDB ダウンロード先
RESULTS_DIR  = "../results/kras_g12c" # ドッキング結果出力先
UNIDOCK2_BIN = "unidock2"              # path to UniDock2 binary
# ------------------------------------------------------------------------------

## 1. セットアップ / Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from IPython.display import display

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis import DockingResult, get_reader
from docking_analysis.preparation.receptor import load_receptor, prepare_receptor
from docking_analysis.preparation.gridbox import gridbox_from_ligand, GridBox
from mdatools.docking.fingerprints.prolif import ProLIFCalculator
from mdatools.docking.visualization.interaction_map import draw_interaction_map
from mdatools.docking.analysis.properties import calculate_properties
from mdatools.docking.analysis.strain import compute_strain_energy, add_strain_to_df
from mdatools.docking.selection.filters import StrainEnergyFilter, apply_filters
from mdatools.docking.selection.pareto import compute_pareto_rank, plot_pareto_2d

RDLogger.DisableLog("rdApp.warning")

data_dir    = Path(DATA_DIR)
results_dir = Path(RESULTS_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 2. PDB 構造の取得 / Download PDB Structures

| PDB ID | リガンド | 段階 | 分解能 | 特徴 |
|--------|---------|------|--------|-----|
| **6N2J** | ARS-853 | ツール化合物 | 2.4 Å | SW2P 標的化の概念実証 |
| **6OIM** | Sotorasib (AMG-510) | FDA 承認 (2021) | 1.9 Å | 世界初 KRAS 直接阻害薬 |

In [ ]:
PDB_IDS = ["6N2J", "6OIM"]

for pdb_id in PDB_IDS:
    dest = data_dir / f"{pdb_id.lower()}.pdb"
    if not dest.exists() or dest.stat().st_size == 0:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading {pdb_id}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print(f"→ {dest}")
    else:
        print(f"{pdb_id}: already exists ({dest})")

## 3. HETATM 確認・Cys12 検出 / Inspect HETATM and Locate Cys12

### KRAS 構造の HETATM 成分

| 成分 | コード | 役割 | 扱い |
|------|-------|------|------|
| **目的リガンド** | ARS/SOT 等 | 解析対象 | 抽出 |
| **GDP** | GDP | KRAS 不活性型安定化 | **保持推奨** ↓ |
| Mg²⁺ | MG | GDP/GTP 配位 | 保持推奨 |
| 結晶充填剤 | GOL, EDO 等 | クライオ保護 | 除去 |

**GDP の扱いについて:**  
SW2P 阻害剤は GDP 結合型 KRAS（inactive 状態）に選択的に結合します。  
`prepare_receptor(keep_resnames=["GDP", "MG"])` で GDP/Mg²⁺ を保持することで、  
より現実的なポケット形状でドッキングを行えます。

In [ ]:
def list_hetatm_residues(pdb_path: Path, min_atoms: int = 1) -> list[dict]:
    WATER_CODES = {"HOH", "WAT", "H2O", "DOD", "D2O"}
    seen = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            res_name = line[17:20].strip()
            chain    = line[21].strip()
            seq_id   = line[22:26].strip()
            key = (chain, res_name, seq_id)
            if res_name not in WATER_CODES:
                seen[key] = seen.get(key, 0) + 1
    return sorted(
        [{"chain": k[0], "residue": k[1], "seqid": k[2], "n_atoms": v}
         for k, v in seen.items() if v >= min_atoms],
        key=lambda x: -x["n_atoms"]
    )


def find_cys_sg(pdb_path: Path, cys_resid: int = 12, chain: str = "A") -> dict | None:
    """Locate a cysteine SG atom in ATOM records."""
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            atom_name = line[12:16].strip()
            res_name  = line[17:20].strip()
            rec_chain = line[21].strip()
            try:
                rec_resid = int(line[22:26].strip())
            except ValueError:
                continue
            if (res_name == "CYS" and rec_resid == cys_resid
                    and atom_name == "SG"
                    and (chain == "*" or rec_chain == chain)):
                return {
                    "resname": "CYS",
                    "resid": rec_resid,
                    "chain": rec_chain,
                    "atom": "SG",
                    "coords": (float(line[30:38]), float(line[38:46]), float(line[46:54])),
                }
    return None


# HETATM 確認
for pdb_id in PDB_IDS:
    residues = list_hetatm_residues(data_dir / f"{pdb_id.lower()}.pdb", min_atoms=4)
    print(f"\n{pdb_id} — 主要 HETATM 残基 (≥4 heavy atoms):")
    for r in residues:
        print(f"  Chain {r['chain']}: {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")

# Cys12 SG の確認
print()
cys_info = {}
for pdb_id in PDB_IDS:
    info = find_cys_sg(data_dir / f"{pdb_id.lower()}.pdb", cys_resid=12)
    if info:
        cys_info[pdb_id] = info
        print(f"{pdb_id}: Cys12 SG found — Chain {info['chain']}, "
              f"coords = ({info['coords'][0]:.2f}, {info['coords'][1]:.2f}, {info['coords'][2]:.2f})")
    else:
        print(f"{pdb_id}: Cys12 NOT found as CYS — check if G12C variant is in the structure")

### 3.1 リガンドコードの設定

上の出力で最も heavy atoms が多い HETATM が目的リガンドです。  
(GDP: ~27 atoms を除外して確認)

- **6N2J** (ARS-853): 出力を確認して設定
- **6OIM** (Sotorasib): 出力を確認して設定

In [ ]:
# ↓ 上のセルの出力を見て残基コードを設定 (GDP/MG 以外の最大 heavy atoms 残基)
LIGAND_CODES = {
    "6N2J": "K9M",   # ARS-853 (RCSB code)
    "6OIM": "MOV",   # Sotorasib/AMG-510 (RCSB code for 6OIM)
}

## 4. リガンド抽出・ウォーヘッドの確認
## Ligand Extraction and Warhead Verification

SW2P 阻害剤のウォーヘッド（acrylamide → Cys12 Michael 付加）を確認します。

**反応機構:**  
`Cys12-SH + CH₂=CH-C(=O)-NHR → Cys12-S-CH₂-CH₂-C(=O)-NHR`

In [ ]:
def extract_ligand_mol(pdb_path: Path, res_code: str, chain: str = "A") -> Chem.Mol | None:
    with open(pdb_path) as f:
        lines = f.readlines()
    hetatm = [l for l in lines
               if l.startswith("HETATM") and l[17:20].strip() == res_code
               and (chain == "*" or l[21].strip() == chain)]
    if not hetatm:
        hetatm = [l for l in lines
                   if l.startswith("HETATM") and l[17:20].strip() == res_code]
    if not hetatm:
        print(f"  WARNING: {res_code} not found in {pdb_path.name}")
        return None
    pdb_block = "".join(hetatm) + "END\n"
    mol = Chem.MolFromPDBBlock(pdb_block, removeHs=True, sanitize=True)
    if mol is None:
        print(f"  WARNING: RDKit could not parse {res_code}")
    return mol


WARHEAD_SMARTS = "[CH2]=[CH]-C(=O)-[NH]"  # acrylamide Michael acceptor
warhead_pattern = Chem.MolFromSmarts(WARHEAD_SMARTS)

ligand_mols = {}
for pdb_id, res_code in LIGAND_CODES.items():
    mol = extract_ligand_mol(data_dir / f"{pdb_id.lower()}.pdb", res_code)
    if mol is None:
        continue
    mol.SetProp("mol_name", res_code)
    mol.SetProp("pdb_id", pdb_id)
    mol.SetProp("pose_rank", "1")
    mol.SetProp("docking_score", "0.0")
    ligand_mols[pdb_id] = mol

    has_warhead = mol.HasSubstructMatch(warhead_pattern)
    label = "ARS-853 (tool)" if pdb_id == "6N2J" else "Sotorasib (FDA approved)"
    print(f"{pdb_id} [{res_code}] {label}: {mol.GetNumAtoms()} heavy atoms, "
          f"acrylamide = {'✓' if has_warhead else '(covalently linked in crystal — see note below)'}")

if ligand_mols:
    mols_list = list(ligand_mols.values())
    legends = [
        f"{'ARS-853 (tool)' if pid == '6N2J' else 'Sotorasib (approved)'}\n{pid}"
        for pid in ligand_mols
    ]
    img = Draw.MolsToGridImage(mols_list, molsPerRow=2, subImgSize=(400, 300), legends=legends)
    display(img)

### 4.1 反応前構造の生成

結晶構造のリガンドは Cys12 との付加体（反応後）です。  
ドッキング入力には**反応前（acrylamide を持つ）構造**が必要です。

**SMILES の取得先:**
- ARS-853: Ostrem et al., *Nature* 2013 (DOI: 10.1038/nature12796) 補足資料  
- Sotorasib: ChEMBL ID `CHEMBL4523630` または PubChem CID `2312459`

In [ ]:
# 反応前（covalent docking 用）構造を SMILES から生成
# 出典: ChEMBL / PubChem / 原著論文

# ARS-853: switch-II pocket binder, chloropyrimidine + acrylamide warhead
# (Approximate SMILES based on Ostrem et al. 2013 — verify from original source)
ARS853_SMILES = "O=C(C=C)Nc1ccc2c(n1)CC[NH+](Cc1ccccc1F)C2"

# Sotorasib / AMG-510 (ChEMBL4523630)
SOTORASIB_SMILES = "CC1(C)CN(c2nc3c(F)cc(Cl)cc3c(=O)[nH]2)C[C@H]1NC(=O)C=C"

pre_reaction_smiles = {
    "6N2J": ARS853_SMILES,
    "6OIM": SOTORASIB_SMILES,
}

DRUG_NAMES = {"6N2J": "ars853", "6OIM": "sotorasib"}

pre_reaction_mols  = {}
input_sdf_paths    = {}

for pdb_id, smi in pre_reaction_smiles.items():
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"{pdb_id}: SMILES parse failed — update SMILES from the reference above")
        continue

    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) == -1:
        print(f"{pdb_id}: 3D embedding failed")
        continue
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("mol_name", LIGAND_CODES.get(pdb_id, pdb_id))
    pre_reaction_mols[pdb_id] = mol

    # SDF 保存
    sdf_path = data_dir / f"{DRUG_NAMES[pdb_id]}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol)
    writer.close()
    input_sdf_paths[pdb_id] = sdf_path

    mol_noh = Chem.RemoveHs(mol)
    has_wh  = mol_noh.HasSubstructMatch(warhead_pattern)
    print(f"{pdb_id} {DRUG_NAMES[pdb_id]}: {mol.GetNumAtoms()} atoms (with H), "
          f"acrylamide = {'✓' if has_wh else '✗ (check SMILES)'}, saved → {sdf_path}")

# 構造表示
mols_display = [Chem.RemoveHs(m) for m in pre_reaction_mols.values()]
legs = [f"{'ARS-853' if pid == '6N2J' else 'Sotorasib'}\n(pre-reaction)" for pid in pre_reaction_mols]
display(Draw.MolsToGridImage(mols_display, molsPerRow=2, subImgSize=(400, 300), legends=legs))

## 5. 受容体準備 / Receptor Preparation

KRAS では **GDP/Mg²⁺ を保持した状態でドッキング**することを推奨します:
- SW2P は GDP 結合型（inactive）構造でのみ開口
- GDP が存在することでポケット形状が実際の状態に近くなる

`prepare_receptor(keep_resnames=["GDP", "MG"])` を使用します。

In [ ]:
receptor_paths = {}
receptor_mols  = {}

# GDP と Mg2+ を保持して受容体を準備
KEEP_COFACTORS = ["GDP", "MG"]

for pdb_id in PDB_IDS:
    raw_pdb = data_dir / f"{pdb_id.lower()}.pdb"
    out_pdb = data_dir / f"{pdb_id.lower()}_receptor.pdb"

    if not out_pdb.exists():
        print(f"Preparing {pdb_id} (keeping GDP/Mg2+)...", end=" ")
        _, rec_mol = prepare_receptor(
            raw_pdb, out_pdb,
            protein_only=True,
            keep_resnames=KEEP_COFACTORS,  # GDP と Mg2+ を保持
        )
        print(f"→ {out_pdb}")
    else:
        print(f"{pdb_id} receptor ready: {out_pdb}")
        rec_mol = load_receptor(out_pdb)

    receptor_paths[pdb_id] = out_pdb
    receptor_mols[pdb_id]  = rec_mol

print("\n受容体準備完了 (GDP/Mg2+ 保持)")
print("注: GDP を除外したい場合は keep_resnames=[] に変更してください")

## 5.1 グリッドボックスの設定
## Grid Box Setup

### SW2P (Switch II Pocket) の特徴

- KRAS 表面の**浅いポケット**（深さ ~5–8 Å）
- ADP/GDP 結合部位とは独立した部位に位置
- Cys12 SG がポケット縁に突出

→ 通常の ATP 結合ポケット（キナーゼの深いポケット）より小さいため、  
`gridbox_from_ligand()` の結果を確認して必要に応じて手動補正します。

In [ ]:
gridboxes = {}
for pdb_id, mol in ligand_mols.items():
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

    # SW2P は浅いポケット → padding は標準 (5 Å) で十分
    grid = gridbox_from_ligand(mol, padding=5.0)

    # SW2P 用: サイズが大きすぎる場合は手動でキャップ
    cx, cy, cz = grid.center
    sx = min(grid.size[0], 28.0)  # SW2P は ~20-28 Å が適切
    sy = min(grid.size[1], 28.0)
    sz = min(grid.size[2], 28.0)
    grid = GridBox(center=(cx, cy, cz), size=(sx, sy, sz))
    gridboxes[pdb_id] = grid

    label = "ARS-853" if pdb_id == "6N2J" else "Sotorasib"
    print(f"\n{pdb_id} [{label}] Switch II Pocket:")
    print(f"  中心 (Å): ({cx:.2f}, {cy:.2f}, {cz:.2f})")
    print(f"  サイズ (Å): ({sx:.2f}, {sy:.2f}, {sz:.2f})")

    # Cys12 SG がグリッドボックス内にあるか確認
    if pdb_id in cys_info:
        x12, y12, z12 = cys_info[pdb_id]["coords"]
        in_box = (abs(x12 - cx) <= sx/2 and abs(y12 - cy) <= sy/2 and abs(z12 - cz) <= sz/2)
        print(f"  Cys12 SG in box: {'✓' if in_box else '⚠ outside — increase padding or adjust center'}",
              f"(Cys12 SG at {x12:.1f}, {y12:.1f}, {z12:.1f})")

## 6. UniDock2 共有結合ドッキング設定と実行
## UniDock2 Covalent Docking Setup and Execution

> ⚠️ このセクションは UniDock2 バイナリと GPU が必要です。

**Cys12 を標的にした共有結合設定:**
```python
covalent_residue_atom_info = [["CYS", 12, "SG"]]
```

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.docking.unidock2 import UniDock2Runner, UniDock2RunConfig

# KRAS G12C 共有結合ドッキング設定
cov_config = UniDock2RunConfig(
    unidock2_binary=UNIDOCK2_BIN,
    covalent_docking=True,
    covalent_residue_atom_info=[["CYS", 12, "SG"]],  # G12C → Cys12
    exhaustiveness=512,
    num_pose=10,
    task="screen",
)

runner = UniDock2Runner(cov_config)

# YAML 設定プレビュー
print("UniDock2RunConfig (KRAS G12C covalent) — YAML preview:")
if gridboxes:
    sample_grid = next(iter(gridboxes.values()))
    yaml_str = runner._build_yaml_config(
        receptor=data_dir / "7m7j_receptor.pdb",
        ligand=data_dir / "sotorasib.sdf",
        grid=sample_grid,
        output_sdf=results_dir / "7m7j_sotorasib_out.sdf",
    )
    print(yaml_str)

In [ ]:
import shutil

docking_results = {}

if shutil.which(UNIDOCK2_BIN) is None:
    print(f"⚠ '{UNIDOCK2_BIN}' not found — skipping docking.")
    print("GPU 環境でのドッキング:")
    print("  docker compose -f docker/docker-compose.yml --profile gpu up")
    print("Section 7 uses crystal structure poses instead.")
else:
    for pdb_id in PDB_IDS:
        if pdb_id not in input_sdf_paths or pdb_id not in gridboxes:
            continue
        out_sdf = results_dir / f"{pdb_id.lower()}_{DRUG_NAMES[pdb_id]}_cov_out.sdf"
        print(f"\nRunning covalent docking: {pdb_id}...")
        try:
            result = runner.run(
                ligand=input_sdf_paths[pdb_id],
                receptor=receptor_paths[pdb_id],
                grid=gridboxes[pdb_id],
                output=out_sdf,
            )
            docking_results[pdb_id] = result
            print(f"  Top score: {result.scores[0]:.2f} kcal/mol")
            print(f"  Poses    : {len(result.poses)}")
        except RuntimeError as e:
            print(f"  Failed: {e}")

## 7. 結合様式の解析（結晶構造ポーズ使用）
## Binding Mode Analysis (Crystal Poses)

In [ ]:
analysis_results = {}
for pdb_id, mol in ligand_mols.items():
    if pdb_id in docking_results:
        analysis_results[pdb_id] = docking_results[pdb_id]
        print(f"{pdb_id}: using docking result")
    else:
        analysis_results[pdb_id] = DockingResult(
            poses=[mol], scores=[0.0],
            source_file=data_dir / f"{pdb_id.lower()}.pdb",
            backend="crystal",
        )
        print(f"{pdb_id}: using crystal structure pose")

### 7.1 ProLIF — SW2P 接触パターン
### ProLIF — Switch II Pocket Contact Pattern

**KRAS SW2P 主要接触残基:**

| 残基 | ループ | ARS-853 | Sotorasib |
|------|-------|---------|----------|
| **Cys12** | P-loop | ✓ (共有結合) | ✓ (共有結合) |
| His95 | SII | △ | ✓ (π-π) |
| Tyr96 | SII | ✓ | ✓ |
| Gln99 | SII | ✓ | ✓ |
| Met72 | SW2P | ✓ | ✓ |
| Lys16 | P-loop | △ | △ |

In [ ]:
calculator = ProLIFCalculator()

fp_results = {}
for pdb_id, result in analysis_results.items():
    rec_mol = receptor_mols.get(pdb_id)
    if rec_mol is None:
        print(f"  {pdb_id}: receptor not loaded — skip")
        continue

    print(f"\n{pdb_id} — ProLIF fingerprint...")
    fp_df = calculator.calculate(result.poses, rec_mol, show_progress=True)
    fp_results[pdb_id] = fp_df

    active_cols = fp_df.columns[fp_df.any()].tolist()
    print(f"  検出された相互作用 ({len(active_cols)} 件):")
    for col in active_cols:
        print(f"    {col}")

In [ ]:
# ARS-853 vs Sotorasib の相互作用比較
if len(fp_results) == 2:
    cols_6n2j = set(fp_results["6N2J"].columns[fp_results["6N2J"].any()])
    cols_7m7j = set(fp_results["6OIM"].columns[fp_results["6OIM"].any()])

    print(f"共通 ({len(cols_6n2j & cols_7m7j)} 件):")
    for c in sorted(cols_6n2j & cols_7m7j): print(f"  {c}")

    print(f"\n6N2J のみ (ARS-853, {len(cols_6n2j - cols_7m7j)} 件):")
    for c in sorted(cols_6n2j - cols_7m7j): print(f"  {c}")

    print(f"\n6OIM のみ (Sotorasib, {len(cols_7m7j - cols_6n2j)} 件):")
    for c in sorted(cols_7m7j - cols_6n2j): print(f"  {c}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
pdb_list = ["6N2J", "6OIM"]
titles   = {
    "6N2J": "ARS-853 (tool compound)\nSW2P proof-of-concept",
    "6OIM": "Sotorasib (FDA approved)\nFull SW2P occupancy",
}

for ax, pdb_id in zip(axes, pdb_list):
    if pdb_id not in fp_results:
        ax.set_title(f"{pdb_id}: data not available")
        continue
    fp_df = fp_results[pdb_id]
    mol   = ligand_mols[pdb_id]
    active_cols = fp_df.columns[fp_df.any()].tolist()
    if active_cols:
        draw_interaction_map(mol=mol, interactions=active_cols,
                             ax=ax, title=f"{pdb_id}\n{titles[pdb_id]}")
    else:
        ax.set_title(f"{pdb_id}: no interactions detected")

plt.tight_layout()
out_fig = results_dir / "kras_g12c_interaction_map.png"
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig}")

## 8. ポーズ品質評価 — 歪みエネルギーと Pareto 多目的評価
## Pose Quality Assessment — Strain Energy and Pareto Analysis

`plot_pareto_2d()` を使って**ドッキングスコア vs 歪みエネルギー**の多目的評価を行います。

- **Pareto rank 1** = スコアと歪みの両方が優れているポーズ（最優先）
- SW2P は浅いポケットのため、通常より歪みが大きくなりやすい（閾値 15 kcal/mol も考慮）

In [ ]:
strain_rows = []
for pdb_id, result in analysis_results.items():
    for i, mol in enumerate(result.poses):
        mol_h = Chem.AddHs(mol)
        if mol_h.GetNumConformers() == 0:
            AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())
        strain = compute_strain_energy(mol_h)
        strain_rows.append({
            "pdb_id"       : pdb_id,
            "ligand"       : LIGAND_CODES.get(pdb_id, pdb_id),
            "pose_rank"    : i + 1,
            "docking_score": result.scores[i] if i < len(result.scores) else None,
            "strain_energy": strain,
        })

strain_df = pd.DataFrame(strain_rows)
print("ポーズ品質サマリー:")
display(strain_df.round(3))

# StrainEnergyFilter (SW2P 阻害剤は少し緩めに 15 kcal/mol)
strain_filter = StrainEnergyFilter(threshold=15.0)
mask     = strain_filter.apply(strain_df)
filtered = strain_df[mask]
print(f"\nStrainEnergyFilter (≤15 kcal/mol): {len(strain_df)} → {len(filtered)} poses")

In [ ]:
# Pareto 多目的評価 (スコア最小化 + 歪みエネルギー最小化)
_required = ["docking_score", "strain_energy"]
if not all(c in strain_df.columns for c in _required):
    print("⚠️  ドッキング結果がないため Pareto プロットをスキップします。")
    print("   ドッキング実行後にこのセルを再実行してください。")
else:
    plot_df = strain_df.dropna(subset=_required).copy()

    if len(plot_df) >= 2:
        pareto_ranks = compute_pareto_rank(
            plot_df,
            objectives=["docking_score", "strain_energy"],
            directions=["min", "min"],
        )
        plot_df["pareto_rank"] = pareto_ranks

        fig = plot_pareto_2d(
            plot_df,
            x_col="docking_score",
            y_col="strain_energy",
            pareto_rank=pareto_ranks,
            output_path=results_dir / "kras_g12c_pareto.png",
        )
        plt.show()
        print("Pareto rank 1 のポーズ（両目標で最良）:")
        display(plot_df[plot_df["pareto_rank"] == 1])
    else:
        print("Pareto プロットには複数ポーズが必要です (ドッキング実行後に再実行)")

## 9. PoseBusters によるポーズ妥当性検証
## Pose Validation with PoseBusters

`validate_poses_posebusters()` で物理化学的妥当性を確認します。

> 注意: `posebusters` はオプション依存パッケージです。  
> インストール: `pip install posebusters`

In [ ]:
try:
    from mdatools.docking.analysis.posebusters import validate_poses_posebusters

    for pdb_id, result in analysis_results.items():
        print(f"\n{pdb_id} — PoseBusters validation...")
        pb_df = validate_poses_posebusters(
            pose_mols=result.poses,
            receptor_pdb=receptor_paths[pdb_id],
            mode="dock",
        )
        n_valid = pb_df["pb_valid"].sum() if "pb_valid" in pb_df.columns else "N/A"
        print(f"  Valid poses: {n_valid} / {len(result.poses)}")
        display(pb_df)

except ImportError:
    print("posebusters is not installed. Install with: pip install posebusters")
    print("Skipping PoseBusters validation.")

## 10. 物性比較と結合様式サマリー
## Properties and Binding Mode Summary

In [ ]:
rows = []
for pdb_id, mol in ligand_mols.items():
    props = calculate_properties(mol)
    rows.append({
        "compound" : f"{LIGAND_CODES[pdb_id]} ({pdb_id})",
        "stage"    : "Tool compound" if pdb_id == "6N2J" else "FDA approved (2021)",
        "warhead"  : "acrylamide → Cys12",
        **props,
    })

props_df = pd.DataFrame(rows).set_index("compound")
display(props_df.round(3))

### 10.1 結合様式サマリー

| 特徴 | ARS-853 (6N2J) | Sotorasib (6OIM) |
|------|----------------|------------------|
| 開発段階 | ツール化合物 | **FDA 承認 (2021)** |
| Cys12 共有結合 | ✓ | ✓ |
| SW2P 占有 | 部分的 | **完全占有** |
| His95 π-π 接触 | △ | ✓ |
| Tyr96 相互作用 | ✓ | ✓ |
| GDP 競合 | なし（別ポケット） | なし |
| 耐性変異 (G12D/V) | × (Cys なし) | × |
| 次世代課題 | Y96D 耐性 | Y96D / G13 変異 |

**意義:**  
ARS-853 は「KRAS G12C が標的になる」ことを証明した先駆的化合物。  
Sotorasib はそこから最適化され、2021 年に初の KRAS 直接阻害薬として FDA 承認。

## 11. 次のステップ / Next Steps

このノートブックで示したワークフロー:

```
PDB ダウンロード → Cys12 検出 + GDP 保持判断
  → 反応前構造生成 (acrylamide warhead)
  → 受容体準備 (GDP/Mg2+ 保持)
  → SW2P グリッドボックス (shallow pocket 補正)
  → UniDock2 covalent docking (Cys12 SG)
  → ProLIF SW2P 接触解析 → Pareto 多目的評価
  → PoseBusters 妥当性確認 → 化合物選択
```

**発展的な解析:**
- `compute_consensus_score()` — 6N2J (ARS-853 pocket) と 6OIM (Sotorasib pocket) に対する consensus
  → Y96D 耐性変異体の PDB（例: 8BD3）との比較
- Adagrasib (MRTX849, 2nd FDA 承認 G12C 阻害剤) との比較
- KRAS G12C 以外の変異 (G12D, G12V) への戦略的アプローチ

---

**関連 Issue:**
- Issue #77: `08_egfr_covalent.ipynb` — EGFR 共有結合 (Cys797)
- Issue #78: `09_kras_g12c_covalent.ipynb` — KRAS G12C (Cys12) ← このノートブック
- Issue #79: `10_mpro_covalent.ipynb` — SARS-CoV-2 Mpro 共有結合